# OPM data access: public ranking and authenticated data

This notebook inspects the public OPM ranking. It shows which information is available without credentials and how the authenticated session is prepared for the next integration step. No password is written in the notebook.

In [1]:
from target_selection import data_source

OPM = data_source.opm
OPM.describe()

{'name': 'opm',
 'kind': 'tom',
 'features': ['targets', 'ranking', 'authenticated_session'],
 'public_ranking': 'https://lensingtom.zah.uni-heidelberg.de/custom_code/prob_list.html',
 'private_data': 'requires an authenticated OPM account'}

## 1. Load the public ranking

The public ranking contains target names, broker links, latest magnitude and band, coordinates, classifier scores, and update times. It is not the same as the private OPM target table.

In [ ]:
ranking = OPM.ranking()
print(f'Public ranking rows: {len(ranking):,}')
display(ranking[['target_id', 'ra_deg', 'dec_deg', 'magnitude_last', 'band_last', 'opm_probability_score', 'opm_updated']].head())

## 2. Apply a simple target filter

The normalized loader can apply the generic magnitude filter. The public ranking's latest magnitude is used for this preliminary selection.

In [ ]:
bright = OPM.load_targets(filters={'magnitude': {'maximum': 20.0}})
print(f'Public ranking targets with latest magnitude <= 20: {len(bright):,}')
display(bright[['target_id', 'magnitude_last', 'band_last', 'ra_deg', 'dec_deg']].head())

## 3. Manual login option

If credentials are not already configured, they can be passed explicitly to `OPM.login`. Do not put real credentials in a committed notebook; use the credential CLI or `.netrc` instead.

In [ ]:
# Run this only after setting the variables outside the notebook:
# import os
# os.environ['TARGET_SELECTION_OPM_USERNAME'] = 'your_username'
# os.environ['TARGET_SELECTION_OPM_PASSWORD'] = 'your_password'
# session = OPM.login()
print('The integration login is shown in the next section.')

## 4. Login used by the integration

The integration uses `OPM.login()`. It obtains credentials from the configured backend, creates an authenticated session and keeps credentials in memory.

In [ ]:
session = OPM.login()
print(type(session).__name__)
print('Authenticated session prepared in memory.')

## 5. Optional explicit login for development

Use this only if credentials are not already configured. Do not put credentials in a committed notebook.

In [ ]:
# session = OPM.login(username='your_username', password='your_password')
# Prefer: target-selection auth set opm
print('Use OPM.login() when credentials are already configured.')

In [ ]:
BASE_URL = OPM.describe()['public_ranking'].split('/custom_code/')[0].rstrip('/')
home = session.get(BASE_URL + '/', timeout=60)
home.raise_for_status()
print('HTTP status:', home.status_code)
print('Final URL:', home.url)
print('Login page detected:', '/accounts/login' in home.url)

## 6. Inspect the authenticated target list

This reads the first target-list page, its table and the search fields exposed by OPM. It does not submit a search form or iterate through all pages.

In [ ]:
import io
import pandas as pd
from bs4 import BeautifulSoup
targets_page = session.get(BASE_URL + '/targets/', timeout=60)
targets_page.raise_for_status()
target_tables = pd.read_html(io.StringIO(targets_page.text))
print(f'Tables found: {len(target_tables)}')
if target_tables:
    display(target_tables[0].head())
target_soup = BeautifulSoup(targets_page.text, 'html.parser')
form_fields = sorted({element.get('name') for element in target_soup.find_all(['input', 'select']) if element.get('name')})
print('Search fields exposed by the page:')
print(form_fields)

## 7. Inspect one target page and its links

The internal numeric page identifier is different from the scientific target name. The example is a public LSST microlensing-candidate page used only for format inspection.

In [ ]:
from urllib.parse import urljoin
OPM_TARGET_PAGE_ID = 13121
target_url = f'{BASE_URL}/targets/{OPM_TARGET_PAGE_ID}/'
target_page = session.get(target_url, timeout=60)
target_page.raise_for_status()
target_soup = BeautifulSoup(target_page.text, 'html.parser')
target_text = ' '.join(target_soup.stripped_strings)
print('Target page:', target_url)
for phrase in ('Microlensing candidate', 'Parameters for target type', 'Photometry', 'Imaging', 'Classifications', 'Analysis'):
    print(f'{phrase}:', phrase.lower() in target_text.lower())
links = []
for anchor in target_soup.find_all('a', href=True):
    href = urljoin(BASE_URL, anchor['href'])
    if any(token in href.lower() for token in ('download', 'lightcurve', 'pylima', '/data/')):
        links.append({'label': anchor.get_text(' ', strip=True), 'url': href})
display(pd.DataFrame(links).drop_duplicates())

## 8. Read one light-curve ZIP in memory

Each file contains JD, magnitude, magnitude uncertainty and a dataset identifier. The ZIP is not saved by this example.

In [ ]:
import zipfile
lightcurve_url = f'{BASE_URL}/download-data/lightcurves/{OPM_TARGET_PAGE_ID}'
lightcurve_response = session.get(lightcurve_url, timeout=60)
lightcurve_response.raise_for_status()
print('HTTP status:', lightcurve_response.status_code)
print('Content type:', lightcurve_response.headers.get('content-type'))
print('Bytes:', len(lightcurve_response.content))
records = []
with zipfile.ZipFile(io.BytesIO(lightcurve_response.content)) as archive:
    print('Files:', archive.namelist())
    for filename in archive.namelist():
        frame = pd.read_csv(archive.open(filename), sep=r'\s+', comment='#', header=None, names=['epoch_hjd', 'magnitude', 'magnitude_err', 'dataset_id'])
        frame['target_id'] = filename.rsplit('/', 1)[-1].rsplit('.', 1)[0]
        frame['band'] = frame['dataset_id'].astype(str).str.extract(r'(?:_|^)([ugrizy])$', expand=False)
        frame['provider'] = 'opm'
        records.append(frame)
opm_observations = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
print(f'Observation rows: {len(opm_observations):,}')
display(opm_observations.head())

## 9. Normalized photometry and implementation status

The adapter now implements authenticated `OPM.photometry(targets)` for rows with `target_id` and `opm_target_page_id`. The raw page and ZIP inspections above remain useful for checking the source format. Event-parameter and image access are still separate future integrations.

In [ ]:
print('Implemented:', ['ranking', 'load_targets', 'login', 'photometry'])
print('Photometry needs target_id and opm_target_page_id from the ranking.')
